In [ ]:
import numpy as np, matplotlib.pyplot as plt
import astropy.units as u
from astropy.time import Time
import yaml
import json
from antpos import utils
from antpos.utils import get_baselines
import dsacalib.constants as ct
from dsacalib.fringestopping import calc_uvw
from astropy.coordinates import SkyCoord

In [ ]:
# sort out constants and setup stuff

bfweights_yaml = "/home/ubuntu/beamformer_weights_2024-01-22T04:45:02.yaml"
mfsconf = "/home/ubuntu/proj/dsa110-shell/dsa110-cnf/config_mfs.yaml"
t2json = "/home/ubuntu/T2_240123aaaf.json"
position = SkyCoord("04:33:03.0","+71:56:43.02",unit=(u.hourangle,u.deg))


#antenna order, snap delays
f = open(bfweights_yaml)
d = yaml.safe_load(f)
f.close()
antenna_order = d["antenna_order"]
eastings = d["eastings"]

#outrigger delays, refmjd
f = open(mfsconf)
d = yaml.safe_load(f)
f.close()
outrigger_delays = d["outrigger_delays"]
refmjd = d["refmjd"]

#time
f = open(t2json)
brst = json.load(f)
f.close()
candname = list(brst.keys())[0]
mjd = brst[candname]['mjds']

#antennas and baselines
df = utils.get_itrf(latlon_center=(ct.OVRO_LAT * u.rad, ct.OVRO_LON * u.rad, ct.OVRO_ALT * u.m))
ant_itrf = np.array([df['dx_m'], df['dy_m'], df['dz_m']]).T
nants_telescope = max(df.index)
df_bls = get_baselines(antenna_order,autocorrs=True,casa_order=False)
bname = df_bls['bname']
blen = np.array([df_bls['x_m'], df_bls['y_m'], df_bls['z_m']]).T

# get all baselines with refant (assume 24 for now - won't work later)
refidxs = []
refant = str(antenna_order[0])
for i, bn in enumerate(bname):
    if refant in bn:
        refidxs += [i]
        

In [ ]:
# calculate per-antenna w-term referenced to refant

_, _, bw = calc_uvw(blen, np.asarray([mjd]), "J2000", position.ra, position.dec)
bw = bw.ravel()
for i, bn in enumerate(bname):
    ant1, ant2 = bn.split('-')
    bw[i] += (outrigger_delays.get(int(ant1), 0) -
              outrigger_delays.get(int(ant2), 0)) * 0.29979245800000004
    a = (outrigger_delays.get(int(ant1), 0) - outrigger_delays.get(int(ant2), 0)) * 0.29979245800000004
    if a>0.:
        print(bn,a)
ant_bw = bw[refidxs]

# np.exp(2j * np.pi / ct.C_GHZ_M * fobs * bws


In [ ]:
# write out
f = open("antbw.dat","w")
for i in np.arange(64):
    f.write(f"{ant_bw[i]}\n")
f.close()

In [ ]:
# test against toolkit assumption
reff = 1.4 # GHz
bm = 89
nant = 64

# toolkit
theta = 1.*(127.-bm*1.)*3.14159265358/10800.
afac = -2.*3.14159265358*(1e9*reff)*theta/299792458.0
w_toolkit = np.zeros(nant,dtype=np.complex64)
for i in np.arange(nant):
    w_toolkit[i] = np.cos(afac*eastings[i])+1j*np.sin(afac*eastings[i])
tabw = theta*np.array(eastings)
    
# with right position
w_full = np.exp(-2j * np.pi / ct.C_GHZ_M * reff * ant_bw)

print(ct.C_GHZ_M)

In [ ]:
plt.figure(figsize=(12,12))
#plt.plot(antenna_order,np.angle(w_toolkit),'o')
#plt.plot(antenna_order,np.angle(w_full)-np.angle(w_toolkit),'x')
plt.plot(antenna_order,ant_bw,'o')
plt.xlim(100,120)
plt.show()